# ДЗ №1. Spark RDD API
### Выполнил: Лёвин В.А.
### Группа: ИУ6-54Б

In [1]:
import csv
import json
import sys
from ast import literal_eval
from datetime import datetime
from math import asin, cos, radians, sin, sqrt
from pathlib import Path

from pyspark import SparkConf, SparkContext

conf = SparkConf().setMaster("local[2]").setAppName("HW1-RDD")
sc = SparkContext.getOrCreate(conf=conf)
sc.pythonExec = sys.executable

26/11/28 21:53:25 WARN Utils: Your hostname, MacBook-Air-mac-4.local resolves to a loopback address: 127.0.0.1; using 192.168.0.116 instead (on interface en0)
26/11/28 21:53:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/28 21:53:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Задание 1. Расстояния между заведениями

### Чтение данных

```python
# Наименование столбцов
['ID', 'Name', 'global_id', 'IsNetObject', 'OperatingCompany', 'TypeObject', 'AdmArea', 'District', 'Address', 'PublicPhone', 'SeatsCount', 'SocialPrivileges', 'Longitude_WGS84', 'Latitude_WGS84', 'geoData']
```

`mapPartitions` разбирает строки каждой части RDD. `cache` сохраняет прочитанные записи для повторных вычислений.


In [2]:
def parse_place(row):
    return {
        "id": int(row[0]),
        "name": row[1],
        "address": row[8],
        "lat": float(row[13]),
        "lon": float(row[12]),
    }

places = (
    sc.textFile("places.csv")
    .mapPartitions(csv.reader)
    .map(parse_place)
    .cache()
)
print(f"Заведений: {places.count()}")

Заведений: 769


### Формула расстояния

$$a = \sin^2\frac{\varphi_2-\varphi_1}{2} + \cos\varphi_1\cos\varphi_2\sin^2\frac{\lambda_2-\lambda_1}{2},$$
$$d = 2R\arcsin\sqrt{a}.$$




In [3]:
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    a = (
        sin((lat2 - lat1) / 2) ** 2
        + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    )
    return 2 * 6_371_000 * asin(sqrt(min(1.0, a)))

### 1. Расстояния от заданной точки

Для каждого заведения считаем расстояние от точки `(55.751244, 37.618423)`. `take(10)` выводит первые записи RDD без сортировки.


In [4]:
def show_places(rows):
    print(f"{'ID':<8} {'Название':<40} {'Расстояние, м':>14} \tАдрес")
    for place, distance in rows:
        print(f"{place['id']:<8} {place['name']:<40} {distance:>14.2f} \t{place['address']}")

point_lat, point_lon = 55.751244, 37.618423
def distance_from_point(place):
    distance = haversine(point_lat, point_lon, place["lat"], place["lon"])
    return place, distance

point_distances = places.map(distance_from_point).cache()
show_places(point_distances.take(10))


ID       Название                                  Расстояние, м 	Адрес
21830    Шоколадница                                     4481.04 	город Москва, Бауманская улица, дом 33/2, строение 1
22366    МУ-МУ                                           4411.45 	город Москва, Бауманская улица, дом 35/1
28608    КОМБИНАТ ПИТАНИЯ МГТУ ИМ.Н.Э.БАУМАНА            4141.57 	город Москва, Бригадирский переулок, дом 3-5
132486   Дом 12                                          2976.78 	город Москва, улица Казакова, дом 8А
22357    Чито-Ра                                         3034.84 	город Москва, улица Казакова, дом 10/2, строение 1
21659    Бар- буфет «Николай»                            2828.73 	город Москва, Старая Басманная улица, дом 5, строение 1
26624    Флорентини                                      2888.58 	город Москва, Старая Басманная улица, дом 9, корпус 1
156563   Beer Gik                                        1565.31 	город Москва, Мясницкая улица, дом 22, строение 1
22129    Погр

### 2. Расстояния между заведениями
Условие `id1 < id2` исключает расстояние до самого себя и повтор пары. 


In [5]:
def is_unique_pair(pair):
    first, second = pair
    return first["id"] < second["id"]

def distance_between_places(pair):
    first, second = pair
    distance = haversine(first["lat"], first["lon"], second["lat"], second["lon"])
    return first, second, distance

def show_pairs(rows):
    print(f"{'Заведение 1':<25} {'Заведение 2':<45} {'Расстояние, м':<14}")
    for first, second, distance in rows:
        name1 = f"{first['id']} {first['name']}"
        name2 = f"{second['id']} {second['name']}"
        print(f"{name1:<25} {name2:<45} {distance:<14.2f}")
        
pairs = places.cartesian(places).filter(is_unique_pair)
pair_distances = pairs.map(distance_between_places).cache()

show_pairs(pair_distances.take(10))

Заведение 1               Заведение 2                                   Расстояние, м 
21830 Шоколадница         22366 МУ-МУ                                   161.32        
21830 Шоколадница         28608 КОМБИНАТ ПИТАНИЯ МГТУ ИМ.Н.Э.БАУМАНА    844.24        
21830 Шоколадница         132486 Дом 12                                 1580.44       
21830 Шоколадница         22357 Чито-Ра                                 1559.11       
21830 Шоколадница         26624 Флорентини                              1594.26       
21830 Шоколадница         156563 Beer Gik                               3025.43       
21830 Шоколадница         22129 Погребок                                3025.43       
21830 Шоколадница         150572 Пробка Гриль                           2936.94       
21830 Шоколадница         154627 TEMPO DI PASTA                         2941.07       
21830 Шоколадница         154750 Хлеб насущный                          2941.07       


26/11/28 21:53:29 WARN BlockManager: Task 3 already completed, not releasing lock for rdd_7_0


### 3. Ближайшие и удалённые пары заведений

`takeOrdered` выбирает наименьшие расстояния, `top` — наибольшие.

In [6]:
def by_distance(row):
    return row[2]

nearest_pairs = pair_distances.takeOrdered(10, key=by_distance)
farthest_pairs = pair_distances.top(10, key=by_distance)

print("10 наиболее близких пар заведений")
show_pairs(nearest_pairs)
print("\n10 наиболее удалённых пар заведений")
show_pairs(farthest_pairs)

places.unpersist()
point_distances.unpersist()
pair_distances.unpersist()

10 наиболее близких пар заведений
Заведение 1               Заведение 2                                   Расстояние, м 
22129 Погребок            156563 Beer Gik                               0.00          
154627 TEMPO DI PASTA     154750 Хлеб насущный                          0.00          
156563 Beer Gik           166443 Kozlovna                               0.00          
22129 Погребок            166443 Kozlovna                               0.00          
156563 Beer Gik           172396 Па-Паэлья                              0.00          
22129 Погребок            172396 Па-Паэлья                              0.00          
120714 Глав Пив Маг       146555 Beermood                               0.00          
143164 Блэк милк          151758 Дабл Би                                0.00          
157308 Jeffreys coffee    170123 Мегобари                               0.00          
143164 Блэк милк          167828 Коленки Пчелы                          0.00          

10 наибо

PythonRDD[7] at RDD at PythonRDD.scala:53

## Задание 2. Рейтинги товаров Amazon

### 1. Средний рейтинг товаров

Из каждого отзыва получаем `(asin, (оценка, 1))`. Через `reduceByKey` складываем оценки и количество отзывов отдельно для каждого товара. Затем делим сумму на количество.


In [7]:
def parse_review(line):
    review = json.loads(line)
    return review["asin"], (float(review["overall"]), 1)


def add_ratings(first, second):
    return first[0] + second[0], first[1] + second[1]


def mean_rating(totals):
    return totals[0] / totals[1]


reviews = sc.textFile("data/Electronics_5.json").map(parse_review)
average_ratings = reviews.reduceByKey(add_ratings).mapValues(mean_rating).cache()

print(f"Товаров с оценками: {average_ratings.count()}")
print(f"{'ASIN':<12} {'Рейтинг':>8}")
for asin, rating in average_ratings.sortByKey().take(10):
    print(f"{asin:<12} {rating:>8.2f}")


Товаров с оценками: 63001
ASIN          Рейтинг


0528881469       2.40
0594451647       4.20
0594481813       4.00
0972683275       4.46
1400501466       3.95
1400501520       4.40
1400501776       4.15
1400532620       3.61
1400532655       3.81
140053271X       3.83


### 2. Названия товаров

`leftOuterJoin` соединяет наборы по `asin` и сохраняет все товары с рассчитанным рейтингом.

In [8]:
def parse_product(line):
    product = literal_eval(line)
    return product["asin"], product.get("title")


def make_product(row):
    asin, (rating, title) = row
    return {"asin": asin, "title": title or "Без названия", "rating": rating}


def show_products(rows):
    print(f"{'ASIN':<12} {'Рейтинг':>8} Название")
    for product in rows:
        print(f"{product['asin']:<12} {product['rating']:>8.2f} {product['title']}")

def by_asin(product):
    return product["asin"]


product_titles = sc.textFile("data/meta_Electronics.json").map(parse_product)
products = average_ratings.leftOuterJoin(product_titles).map(make_product).cache()
show_products(products.takeOrdered(10, key=by_asin))


ASIN          Рейтинг Название
0528881469       2.40 Rand McNally 528881469 7-inch Intelliroute TND 700 Truck GPS
0594451647       4.20 Barnes &amp; Noble HDTV Adapter Kit for NOOK HD and NOOK HD+
0594481813       4.00 Barnes &amp; Noble OV/HB-ADP Universal Power Kit
0972683275       4.46 VideoSecu 24&quot; Long Arm TV Wall Mount Low Profile Full Motion Cantilever Swing &amp; Tilt wall bracket for most 22&quot; to 55&quot; LED LCD TV Monitor Flat Panel Screen VESA 200x200 400x400 up to 600x400mm - Articulating Arm Extend up to 24&quot; MAH
1400501466       3.95 Barnes &amp; Noble NOOK Tablet 16gb (Color, BNTV250)
1400501520       4.40 NOOK HD 7&quot; 16GB Tablet
1400501776       4.15 Barnes and Noble NOOK 7&quot; 8GB WiFi Tablet
1400532620       3.61 Barnes &amp; Noble Nook eReader - no 3G
1400532655       3.81 Barnes &amp; Noble Nook Color eReader
140053271X       3.83 Barnes &amp; Noble Nook Simple Touch eBook Reader (Wi-Fi Only)


### 3. Товары с рейтингом ниже 3

Фильтруем по среднему рейтингу без округления. Выводим 10 товаров с наименьшим рейтингом

In [9]:
def is_low_rated(product):
    return product["rating"] < 3


def by_rating(product):
    return product["rating"], product["asin"]


low_rated_products = products.filter(is_low_rated).cache()
print(f"Товаров с рейтингом ниже 3: {low_rated_products.count()}")
show_products(low_rated_products.takeOrdered(10, key=by_rating))


Товаров с рейтингом ниже 3: 2625
ASIN          Рейтинг Название
B00000JBIA       1.00 Agfa ePhoto SMILE 0.2MP Digital Camera
B000F1ORW6       1.00 GE 24746 Futura HDTV Ready Antenna
B000H13L4Y       1.00 ATI TV Wonder 200 PCI Video Card w/PVR Capabilities
B000NNFS4C       1.00 RCA DRC8335 DVD Recorder &amp; VCR Combo With Built-In Tuner
B00111JODG       1.00 StarTech HDMISPL1HH 1 feet Standard HDMI Cable - 1x HDMI (M) to 2x HDMI (F) (Discontinued by Manufacturer)
B0013WI5SS       1.00 Dynex-DX-AP100 Adapter Mini DVI to Mini-DIN
B001BLN91Y       1.00 Maxtor OneTouch 4 - 1 TB USB 2.0 Desktop External Hard Drive STM310005OTA3E1-RK
B001T9N0R6       1.00 Zeikos 57-in-1 USB 2.0 Flash Memory Card Reader ZE-CR201
B003KIQTXG       1.00 NEEWER&reg; Photographic Barn Door &amp; Honeycomb Grid &amp; Gel Set for Alienbees Alienbee Flash
B004GGRPGG       1.00 Rapid USB Charger Adapters include: Wall and Car + USB Cable for Samsung Galaxy TAB (P1000)


### 4. Сохранение результата

Сохраняем весь RDD товаров с рейтингом ниже 3.


In [10]:
def product_to_json(product):
    return json.dumps(product, ensure_ascii=False)

run_name = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
output_path = Path("output") / f"low_rated_products_{run_name}"
low_rated_products.map(product_to_json).saveAsTextFile(str(output_path))
print(f"Результат сохранён в {output_path}")

average_ratings.unpersist()
products.unpersist()
low_rated_products.unpersist()

Результат сохранён в output/low_rated_products_20261128_215355_899729


PythonRDD[37] at RDD at PythonRDD.scala:53

## Задание 3. Сходство фильмов

### Чтение данных

Для каждого фильма собираем словарь `{userId: rating}`. Это его вектор оценок: отсутствующие оценки считаем нулевыми.


In [11]:
def is_rating(row):
    return row[0] != "userId"


def parse_rating(row):
    return int(row[1]), (int(row[0]), float(row[2]))


ratings = (
    sc.textFile("ml-latest-small/ratings.csv")
    .mapPartitions(csv.reader)
    .filter(is_rating)
    .map(parse_rating)
)
movie_vectors = ratings.groupByKey().mapValues(dict).cache()
print(f"Фильмов с оценками: {movie_vectors.count()}")


Фильмов с оценками: 9724


### 1. Косинусное сходство

Функция принимает два словаря оценок и возвращает одно число. Сама вычисляет скалярное произведение и длины векторов.

$$\operatorname{sim}(i,j)=\frac{\sum_u r_{ui}r_{uj}}{\sqrt{\sum_u r_{ui}^2}\sqrt{\sum_u r_{uj}^2}}.$$



In [12]:
def cosine_similarity(first, second):
    dot_product = sum(rating * second.get(user, 0) for user, rating in first.items())
    norm1 = sqrt(sum(rating ** 2 for rating in first.values()))
    norm2 = sqrt(sum(rating ** 2 for rating in second.values()))
    return dot_product / (norm1 * norm2)


first_movie, second_movie = movie_vectors.take(2)
print(f"Сходство фильмов {first_movie[0]} и {second_movie[0]}: "
      f"{cosine_similarity(first_movie[1], second_movie[1]):.6f}")


Сходство фильмов 6 и 50: 0.456850


### 2. RDD сходств с фильмом 589

Получаем вектор фильма `589` и сравниваем его с каждым другим фильмом. Элемент результата — `(movieId, similarity)`.


In [13]:
target_movie = 589


def is_target_movie(row):
    return row[0] == target_movie


def is_other_movie(row):
    return row[0] != target_movie


def compare_with_target(row):
    movie_id, vector = row
    return movie_id, cosine_similarity(target_vector, vector)


target_vector = movie_vectors.filter(is_target_movie).values().first()
similarities = movie_vectors.filter(is_other_movie).map(compare_with_target).cache()
print(f"Сходств с фильмом {target_movie}: {similarities.count()}")


Сходств с фильмом 589: 9723


### 3. Названия фильмов

Добавляем названия к RDD по `movieId`.


In [14]:
def is_movie(row):
    return row[0] != "movieId"


def parse_movie(row):
    return int(row[0]), row[1]


def add_title(row):
    movie_id, (similarity, title) = row
    return {"movieId": movie_id, "title": title, "similarity": similarity}


titles = (
    sc.textFile("ml-latest-small/movies.csv")
    .mapPartitions(csv.reader)
    .filter(is_movie)
    .map(parse_movie)
)
movies_with_similarity = similarities.join(titles).map(add_title)
movies_with_similarity.take(3)

[{'movieId': 216,
  'title': 'Billy Madison (1995)',
  'similarity': 0.23031120299047797},
 {'movieId': 260,
  'title': 'Star Wars: Episode IV - A New Hope (1977)',
  'similarity': 0.5900777578543723},
 {'movieId': 296,
  'title': 'Pulp Fiction (1994)',
  'similarity': 0.6102841093789623}]

### 4. Топ-10 похожих фильмов


In [15]:
def by_similarity(movie):
    return movie["similarity"]


def show_movies(rows):
    print(f"{'movieId':<8} {'Сходство':>10} Название")
    for movie in rows:
        print(f"{movie['movieId']:<8} {movie['similarity']:>10.6f} {movie['title']}")


top_movies = movies_with_similarity.top(10, key=by_similarity)
show_movies(top_movies)

movie_vectors.unpersist()
similarities.unpersist()

movieId    Сходство Название
480        0.719983 Jurassic Park (1993)
1240       0.695724 Terminator, The (1984)
110        0.659827 Braveheart (1995)
592        0.645603 Batman (1989)
457        0.637561 Fugitive, The (1993)
377        0.630092 Speed (1994)
1196       0.618530 Star Wars: Episode V - The Empire Strikes Back (1980)
380        0.611164 True Lies (1994)
296        0.610284 Pulp Fiction (1994)
356        0.600886 Forrest Gump (1994)


PythonRDD[54] at RDD at PythonRDD.scala:53